In [ ]:
# Data preprocessing
import pandas                  as     pd
import numpy                   as     np
import missingno               as     msno

from   sklearn.model_selection import train_test_split
import statsmodels.api         as     sm
import scipy.stats             as     stats

# Visualization
import seaborn                 as     sns
import matplotlib.pyplot       as     plt
sns.set_style("whitegrid")

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import random  # Set the global random seed
# Set the global random seed
def set_global_seed(seed_value):
    random.seed(seed_value)  # Set the Python random seed
    np.random.seed(seed_value)  # Set the NumPy random seed

set_global_seed(66)

In [ ]:
# Set the Matplotlib font to Times New Roman
plt.rcParams['font.family'] = 'Times New Roman'

In [ ]:
data = pd.read_csv('')

In [ ]:
data.shape

In [ ]:
x = data.drop(columns=[''])
y = data['']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Split the data into training and test sets
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.2, random_state=42)

# Prediction

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.model_selection import KFold, cross_validate

In [ ]:
# -------------------- 1. Data preparation --------------------
# Standardization
scaler = StandardScaler().fit(train_x)
train_x_scaled = scaler.transform(train_x)
test_x_scaled  = scaler.transform(test_x)

# 5-fold cross-validation object
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Define models and hyperparameter search spaces
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

In [ ]:
model_spaces = {
    'LinearRegression': (
        LinearRegression(),
        {
            'fit_intercept': Categorical([True, False]),
            'copy_X':        Categorical([True, False]),
            'positive':      Categorical([True, False])
        }
    ),
    'Ridge': (
        Ridge(),
        {
            'alpha':         Real(0.01, 10, prior='log-uniform'),
            'fit_intercept': Categorical([True, False]),
            'solver':        Categorical(['auto','svd','cholesky','lsqr','sparse_cg','sag','saga'])
        }
    ),
    'Lasso': (
        Lasso(max_iter=5000),
        {
            'alpha':         Real(0.001, 1.0, prior='log-uniform'),
            'fit_intercept': Categorical([True, False]),
            'selection':     Categorical(['cyclic','random'])
        }
    ),
    'ElasticNet': (
        ElasticNet(max_iter=5000),
        {
            'alpha':        Real(0.001, 1.0, prior='log-uniform'),
            'l1_ratio':     Real(0.01, 0.99),
            'fit_intercept':Categorical([True, False])
        }
    ),
    'SVR': (
        SVR(),
        {
            'kernel':  Categorical(['linear','poly','rbf']),
            'C':       Real(0.1, 100, prior='log-uniform'),
            'epsilon': Real(0.01, 1.0, prior='log-uniform'),
            'gamma':   Real(0.001, 1.0, prior='log-uniform')
        }
    ),
    'RandomForest': (
        RandomForestRegressor(random_state=42),
        {
            'n_estimators':      Integer(50,300),
            'max_depth':         Integer(5,30),
            'min_samples_split': Integer(2,20),
            'min_samples_leaf':  Integer(1,10),
            'max_features':      Categorical(['sqrt','log2', None])
        }
    ),
    'GradientBoosting': (
        GradientBoostingRegressor(random_state=42),
        {
            'n_estimators':     Integer(50,300),
            'learning_rate':    Real(0.01,0.3, prior='log-uniform'),
            'max_depth':        Integer(3,10),
            'min_samples_split':Integer(2,20),
            'min_samples_leaf': Integer(1,10),
            'subsample':        Real(0.5,1.0)
        }
    ),
    'ExtraTrees': (
        ExtraTreesRegressor(random_state=42),
        {
            'n_estimators':      Integer(50,300),
            'max_depth':         Integer(5,30),
            'min_samples_split': Integer(2,20),
            'min_samples_leaf':  Integer(1,10),
            'max_features':      Categorical(['sqrt','log2', None])
        }
    ),
    'XGBoost': (
        XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1),
        {
            'n_estimators':      Integer(50,300),
            'max_depth':         Integer(3,10),
            'learning_rate':     Real(0.01,0.3, prior='log-uniform'),
            'subsample':         Real(0.5,1.0),
            'colsample_bytree':  Real(0.5,1.0),
            'gamma':             Real(1e-6,5, prior='log-uniform')
        }
    )
}


In [ ]:
import time

In [ ]:
# Bayesian optimization and 5-fold cross-validation evaluation
best_estimators = {}
cv_metrics = {}
scoring = {'MSE': 'neg_mean_squared_error', 'MAE': 'neg_mean_absolute_error', 'R2': 'r2'}

for name, (estimator, space) in model_spaces.items():
    print(f"\n>> Optimizing {name}...")
    bayes = BayesSearchCV(
        estimator=estimator,
        search_spaces=space,
        n_iter=30,
        cv=kf,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    start_time = time.time()
    bayes.fit(train_x_scaled, train_y)
    print(f"  Best params for {name}: {bayes.best_params_} ({time.time() - start_time:.1f}s)")
    best_estimators[name] = bayes.best_estimator_

    cv_res = cross_validate(
        bayes.best_estimator_, train_x_scaled, train_y,
        cv=kf, scoring=scoring, return_train_score=False
    )
    cv_metrics[name] = {
        'MSE':  -cv_res['test_MSE'].mean(),
        'RMSE': np.sqrt(-cv_res['test_MSE']).mean(),
        'MAE':  -cv_res['test_MAE'].mean(),
        'R²':   cv_res['test_R2'].mean()
    }
    print(f"  5-fold CV for {name}: MSE={cv_metrics[name]['MSE']:.4f}, RMSE={cv_metrics[name]['RMSE']:.4f}, MAE={cv_metrics[name]['MAE']:.4f}, R²={cv_metrics[name]['R²']:.4f}")

In [ ]:
# Summarize results and create visualizations
results_df = pd.DataFrame(cv_metrics).T
print("\n=== 5-fold CV Summary ===")
print(results_df)

plt.figure(figsize=(10, 6))
for metric in ['RMSE', 'MAE', 'R²']:
    sns.barplot(x=results_df.index, y=results_df[metric], label=metric)
plt.xticks(rotation=45, ha='right')
plt.title('Comparison of 5-fold CV metrics across models')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.cm as cm

In [ ]:
# --------------------------------------------
# Test set predictions and Taylor diagram
# --------------------------------------------
# Generate predictions
predictions = {name: model.predict(test_x_scaled) for name, model in best_estimators.items()}

# Taylor diagram function in English

def taylor_diagram(ref, model_preds, fig=None, rect=111, ref_label='Reference'):
    std_ref = np.std(ref)
    if fig is None:
        fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(rect, projection='polar')
    ax.set_thetamin(0)
    ax.set_thetamax(90)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    max_std = std_ref * 1.5
    ax.set_ylim(0, max_std)
    ax.plot(0, std_ref, 'ko', label=ref_label, markersize=12, markeredgewidth=2)

    colors = cm.viridis(np.linspace(0.1, 0.9, len(model_preds)))
    best_name, best_corr = None, -np.inf

    for (name, preds), color in zip(model_preds.items(), colors):
        std_mod = np.std(preds)
        corr = np.corrcoef(ref, preds)[0, 1]
        theta = np.degrees(np.arccos(corr))
        if corr > best_corr:
            best_corr, best_name = corr, name
        ax.plot(np.radians(theta), std_mod, 'o', label=name, color=color,
                markersize=8, markeredgewidth=1.5)

    if best_name:
        best_theta = np.degrees(np.arccos(best_corr))
        ax.plot(np.radians(best_theta), np.std(model_preds[best_name]), 'D',
                label=f"Best model: {best_name}", color='red', markersize=12, markeredgewidth=2)

    corr_vals = np.linspace(0, 1, 6)
    angles = np.degrees(np.arccos(corr_vals))
    ax.set_thetagrids(angles, labels=[f"r={c:.2f}" for c in corr_vals])

    rad_ticks = np.linspace(0, max_std, 6)[1:]
    ax.set_rticks(rad_ticks)
    ax.set_yticklabels([f"{r:.2f}" for r in rad_ticks])

    # RMSD contours
    rs = np.linspace(0, max_std, 100)
    ts = np.linspace(0, np.pi/2, 100)
    RR, TT = np.meshgrid(rs, ts)
    RMSD = np.sqrt(std_ref**2 + RR**2 - 2*std_ref*RR*np.cos(TT))
    levels = np.linspace(0, max_std, 6)[1:]
    ax.contour(TT, RR, RMSD, levels=levels, colors='gray',
               linestyles='--', linewidths=0.8)

    plt.title('Taylor Diagram: Model Comparison', fontsize=14, fontfamily='Times New Roman', pad=15)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    plt.tight_layout()
    return fig

# Plot Taylor diagram
fig = taylor_diagram(test_y.values, predictions)
plt.show()
